# 🧠 MS Lesion Segmentation: Training Notebook
### Stage 1: Infrastructure & Data Ingestion

This notebook is designed to handle training on **Google Colab (T4 GPU)** or your **Local Machine** automatically.

## ⚙️ 1.0 Global Configuration & Hardware Switch
This cell detects where the code is running and sets the correct paths automatically.

In [ ]:
import os, sys, torch

# ─── MANUAL OVERRIDE ─────────────────────────────────────────────
ENVIRONMENT = "local"   # Change to "colab" if using Colab
# ─────────────────────────────────────────────────────────────────────────────

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR       = "/content"
    DRIVE_ROOT     = "/content/drive/MyDrive"
    DATA_ZIP_DIR   = os.path.join(DRIVE_ROOT, "brain_dataset")   # where ZIPs live
    MODEL_SAVE_DIR = os.path.join(DRIVE_ROOT, "MS_Project/models")
    CACHE_DIR      = "/content/p_cache"
else:
    BASE_DIR       = "."
    DATA_ZIP_DIR   = None          # no ZIPs — data already extracted locally
    MODEL_SAVE_DIR = "./models"    # save locally; rclone will sync to Drive
    CACHE_DIR      = "./data/p_cache"

DATA_DIR      = os.path.join(BASE_DIR, "data/raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")

for p in [DATA_DIR, PROCESSED_DIR, MODEL_SAVE_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0  # single-process loading: stable on all OS/notebook environments
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True  # ~10-15% speedup for fixed patch sizes
print(f"Environment : {ENVIRONMENT}")
print(f"Device      : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"CPU workers : {NUM_WORKERS}")

## 🛠 1.1 Environment Setup
Install core medical AI libraries and mount Google Drive if in Colab.

In [ ]:
if ENVIRONMENT == "colab":
    import subprocess
    subprocess.run(["pip", "install", "-q", "monai[all]", "SimpleITK", "nibabel", "tqdm"], check=True)
print("✅ Workspace ready.")

## 📊 1.2 Data Ingestion (Hybrid Strategy)
We pull the ZIP files from your **Google Drive** and extract them to **local storage** for maximum training speed.

In [ ]:
import glob, zipfile, subprocess

if ENVIRONMENT == "colab" and DATA_ZIP_DIR:
    for category in ["MSLesSeg", "Mendeley", "Long-MR-MS"]:
        local_folder = os.path.join(DATA_DIR, category)
        os.makedirs(local_folder, exist_ok=True)
        for zip_path in glob.glob(os.path.join(DATA_ZIP_DIR, category, "*.zip")):
            print(f"Extracting {os.path.basename(zip_path)}...")
            subprocess.run(["unzip", "-q", "-o", zip_path, "-d", local_folder], check=True)
    print("✅ Extraction complete.")
else:
    # Extract any ZIPs placed by Data_Downloader (local mode); no-op if already extracted
    for category in ["MSLesSeg", "Mendeley", "Long-MR-MS"]:
        dest = os.path.join(DATA_DIR, category)
        for zip_path in glob.glob(os.path.join(dest, "*.zip")):
            flag = zip_path.replace(".zip", ".extracted")
            if not os.path.exists(flag):
                print(f"📦 Extracting {os.path.basename(zip_path)}...")
                with zipfile.ZipFile(zip_path, 'r') as z:
                    z.extractall(dest)
                open(flag, 'w').close()
    print(f"✅ Data ready at: {DATA_DIR}")

## 🔍 1.2.1 Directory Mapping (Diagnostic)
Run this cell to see exactly where your FLAIR and Mask files are located inside the subfolders.

In [ ]:
import os

def map_directory_structure(root_dir, max_depth=3):
    print(f"🔍 Mapping structure for: {root_dir}\n")
    for root, dirs, files in os.walk(root_dir):
        depth = root.replace(root_dir, '').count(os.sep)
        if depth <= max_depth:
            indent = '  ' * depth
            print(f"{indent}📂 {os.path.basename(root)}/")
            if files:
                for f in files[:3]:
                    print(f"{indent}  📄 {f}")
                if len(files) > 3:
                    print(f"{indent}  ... ({len(files)-3} more files)")

map_directory_structure(DATA_DIR)

## 📂 1.2.2 Data Discovery & Pairing
This cell pairs MRI scans with their Ground Truth masks for training across all datasets.

In [ ]:
import glob

def create_data_list(raw_dir):
    data_list = []
    
    # 1. Long-MR-MS
    long_path = os.path.join(raw_dir, 'Long-MR-MS')
    if os.path.exists(long_path):
        for p_folder in glob.glob(os.path.join(long_path, 'patient*')):
            gt = glob.glob(os.path.join(p_folder, '*_gt.nii.gz'))
            flairs = glob.glob(os.path.join(p_folder, '*_FLAIRreg.nii.gz'))
            if gt and flairs:
                for f in flairs:
                    data_list.append({'image': f, 'label': gt[0]})

    # 2. Mendeley
    mendeley_path = os.path.join(raw_dir, 'Mendeley')
    if os.path.exists(mendeley_path):
        flair_imgs = [f for f in glob.glob(os.path.join(mendeley_path, '**/*-Flair.nii'), recursive=True) 
                      if "LESIONSEG" not in f.upper()]
        for f in flair_imgs:
            p_id = os.path.basename(f).split('-')[0]
            mask = os.path.join(os.path.dirname(f), f"{p_id}-LesionSeg-Flair.nii")
            if os.path.exists(mask):
                data_list.append({'image': f, 'label': mask})

    # 3. MSLesSeg
    msles_path = os.path.join(raw_dir, 'MSLesSeg')
    if os.path.exists(msles_path):
        all_nii = glob.glob(os.path.join(msles_path, '**/*.nii*'), recursive=True)
        mask_map = {}
        for m in all_nii:
            if "_MASK" in m.upper():
                parts = os.path.basename(m).split('_')
                if len(parts) >= 2: mask_map[f"{parts[0]}_{parts[1]}"] = m
        
        flairs = [f for f in all_nii if "FLAIR" in f.upper() and "_MASK" not in f.upper()]
        for f in flairs:
            parts = os.path.basename(f).split('_')
            if len(parts) >= 2:
                key = f"{parts[0]}_{parts[1]}"
                if key in mask_map: data_list.append({'image': f, 'label': mask_map[key]})
    
    print(f"✅ Total Aligned Pairs: {len(data_list)}")
    return data_list

train_files = create_data_list(DATA_DIR)

## 🚀 1.3 GPU-Accelerated Preprocessing
Standards volumes to 1mm isotropic spacing using MONAI GPU transforms.

In [ ]:
import SimpleITK as sitk
import numpy as np
from monai.transforms import (Compose, LoadImage, EnsureChannelFirst, Orientation, Spacing, ScaleIntensity)

def get_gpu_preprocessor(is_mask=False):
    return Compose([
        LoadImage(image_only=True),
        EnsureChannelFirst(),
        Orientation(axcodes="RAS"),
        Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear" if not is_mask else "nearest"),
        ScaleIntensity() if not is_mask else Compose([]),
    ])

def apply_n4_cpu(img_path):
    img = sitk.ReadImage(img_path)
    mask = sitk.OtsuThreshold(img, 0, 1)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrector.SetMaximumNumberOfIterations([20, 20, 20])
    return corrector.Execute(img, mask)

def apply_skull_strip_cpu(sitk_img):
    """
    Lightweight skull strip via Otsu + morphological operations.
    Removes skull/scalp voxels so normalization is computed on brain-only tissue.
    This avoids skewed Z-score from non-brain voxels and improves cross-scanner generalization.
    No external tools needed — pure SimpleITK.
    """
    # Otsu threshold to get rough brain mask
    otsu = sitk.OtsuThreshold(sitk_img, 0, 1)
    # Fill holes and close gaps with morphological operations
    filled = sitk.BinaryFillhole(otsu)
    closed = sitk.BinaryMorphologicalClosing(filled, [3, 3, 3])
    # Keep only largest connected component (brain, not scattered skull fragments)
    labeled = sitk.ConnectedComponent(closed)
    sorted_comp = sitk.RelabelComponent(labeled, sortByObjectSize=True)
    brain_mask = sitk.Equal(sorted_comp, 1)
    # Apply mask: zero out non-brain voxels
    mask_float = sitk.Cast(brain_mask, sitk.sitkFloat32)
    stripped = sitk.Multiply(sitk.Cast(sitk_img, sitk.sitkFloat32), mask_float)
    return stripped

print("✅ Preprocessor logic loaded.")

## ⚡ 1.3.1 Bulk Parallel GPU Preprocessing
Standardizes all patient pairs into the `/processed` folder.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import nibabel as nib, numpy as np, SimpleITK as sitk, torch
from monai.transforms import Compose, EnsureChannelFirst, Orientation, Spacing, ScaleIntensity
from monai.data import MetaTensor

GPU_BATCH = 16
N_THREADS = 4 if ENVIRONMENT == "colab" else min(16, max(1, os.cpu_count() - 2))

# ── Fast-path: check disk first — skip N4 entirely if all pairs already exist ──
processed_train_files = []
_i = 0
while True:
    _d = os.path.join(PROCESSED_DIR, f"sub-{_i:03d}")
    if not os.path.exists(_d):
        break
    _img, _lbl = os.path.join(_d, "image.nii.gz"), os.path.join(_d, "label.nii.gz")
    if os.path.exists(_img) and os.path.exists(_lbl):
        processed_train_files.append({"image": _img, "label": _lbl})
    _i += 1

if len(processed_train_files) == len(train_files):
    print(f"Skipping preprocessing — {len(processed_train_files)} pairs already on disk.")
else:
    remaining = len(train_files) - len(processed_train_files)
    print(f"{len(processed_train_files)} cached, {remaining} still need processing — running pipeline...")

    # ── Phase A: CPU worker — N4 only, returns numpy arrays ──────────────────
    def run_n4(args):
        idx, pair = args
        try:
            raw  = sitk.ReadImage(pair['image'])
            mask = sitk.OtsuThreshold(raw, 0, 1)
            cor  = sitk.N4BiasFieldCorrectionImageFilter()
            cor.SetMaximumNumberOfIterations([20, 20, 20])
            corrected = cor.Execute(raw, mask)
            stripped  = apply_skull_strip_cpu(corrected)
            img_arr   = sitk.GetArrayFromImage(stripped).astype(np.float32)
            lbl_raw   = sitk.ReadImage(pair['label'])
            lbl_arr   = sitk.GetArrayFromImage(lbl_raw).astype(np.float32)
            spacing   = list(reversed(raw.GetSpacing()))
            return idx, img_arr, lbl_arr, spacing, pair
        except Exception as e:
            return idx, None, None, None, str(e)

    # ── Phase B: GPU batch transform ─────────────────────────────────────────
    img_gpu_tf = Compose([Orientation(axcodes="RAS"),
                           Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
                           ScaleIntensity()])
    lbl_gpu_tf = Compose([Orientation(axcodes="RAS"),
                           Spacing(pixdim=(1.0, 1.0, 1.0), mode="nearest")])

    def gpu_process_batch(batch_items):
        results = []
        for idx, img_arr, lbl_arr, spacing, pair in batch_items:
            subj_dir = os.path.join(PROCESSED_DIR, f"sub-{idx:03d}")
            img_out  = os.path.join(subj_dir, "image.nii.gz")
            lbl_out  = os.path.join(subj_dir, "label.nii.gz")
            if os.path.exists(img_out) and os.path.exists(lbl_out):
                results.append({'image': img_out, 'label': lbl_out})
                continue
            os.makedirs(subj_dir, exist_ok=True)
            try:
                sp     = spacing
                affine = torch.diag(torch.tensor([sp[0], sp[1], sp[2], 1.0], dtype=torch.float64))
                if img_arr.ndim == 3:
                    img_arr = np.expand_dims(img_arr, 0)
                if lbl_arr.ndim == 3:
                    lbl_arr = np.expand_dims(lbl_arr, 0)
                img_meta = MetaTensor(torch.tensor(img_arr), affine=affine)
                lbl_meta = MetaTensor(torch.tensor(lbl_arr), affine=affine)
                img_t = img_gpu_tf(img_meta.to(device)).cpu()
                lbl_t = lbl_gpu_tf(lbl_meta.to(device)).cpu()
                nib.save(nib.Nifti1Image(img_t.numpy().squeeze(), np.eye(4)), img_out)
                nib.save(nib.Nifti1Image(lbl_t.numpy().squeeze(), np.eye(4)), lbl_out)
                results.append({'image': img_out, 'label': lbl_out})
            except Exception as e:
                print(f"  ERROR sub-{idx:03d}: {e}")
        return results

    # ── Orchestrate ───────────────────────────────────────────────────────────
    pending_batch = []
    with ThreadPoolExecutor(max_workers=N_THREADS) as pool:
        futures = {pool.submit(run_n4, (i, p)): i for i, p in enumerate(train_files)}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="N4 + GPU pipeline"):
            idx, img_arr, lbl_arr, spacing, pair_or_err = fut.result()
            if img_arr is None:
                print(f"  ERROR sub-{idx:03d}: {pair_or_err}")
                continue
            pending_batch.append((idx, img_arr, lbl_arr, spacing, pair_or_err))
            if len(pending_batch) >= GPU_BATCH:
                processed_train_files.extend(gpu_process_batch(pending_batch))
                pending_batch = []
    if pending_batch:
        processed_train_files.extend(gpu_process_batch(pending_batch))
    print(f"Done: {len(processed_train_files)} pairs ready.")

## 🧪 1.4 Architecture: 3D Residual U-Net

In [ ]:
from monai.networks.nets import UNet

def get_model(in_channels=None, out_channels=1):
    ch = in_channels if in_channels is not None else N_CHANNELS
    return UNet(
        spatial_dims=3, in_channels=ch, out_channels=out_channels,
        channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
        num_res_units=2, norm="INSTANCE", act="LEAKYRELU", dropout=0.1
    )

# N_CHANNELS is set by the ablation cell (cell 20) via create_multimodal_data_list().
# get_model() must only be called after that cell has run.
print("Architecture defined. Call get_model() after the ablation cell sets N_CHANNELS.")

## 📉 1.5 Loss & Metrics

In [ ]:
from monai.losses import DiceLoss
import torch.nn as nn
from scipy.ndimage import label

class MSLesionLoss(nn.Module):
    def __init__(self, alpha=1.0, weight=10.0):
        super(MSLesionLoss, self).__init__()
        self.alpha, self.dice_loss = alpha, DiceLoss(sigmoid=True, squared_pred=True)
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([weight]).to(device))
    def forward(self, input, target):
        return self.dice_loss(input, target) + self.alpha * self.bce_loss(input, target)

print("✅ Loss logic ready.")

In [ ]:
# ── Multi-Modal Ablation Configuration ───────────────────────────────────────
# Controls which MRI modalities are loaded as input channels.
# Ablation A: FLAIR only        (in_channels=1) — current default, fastest
# Ablation B: FLAIR + T1        (in_channels=2) — +3-6% Dice expected
# Ablation C: FLAIR + T1 + T2   (in_channels=3) — marginal gain over B
#
# NOTE: T1/T2 files must exist alongside FLAIR in each dataset folder.
# If a patient is missing T1 or T2, that pair is silently dropped.
# ─────────────────────────────────────────────────────────────────────────────
ABLATION = "A"   # Change to "B" or "C" to enable multi-modal input

def create_multimodal_data_list(raw_dir, ablation=ABLATION):
    """
    Extends create_data_list() to optionally load T1/T2 alongside FLAIR.
    Returns dicts with keys: 'image' (list of paths), 'label'.
    For ablation A, 'image' is a single path string (backward compatible).
    """
    import glob as _glob
    data_list = []

    def _find_modality(folder, pattern):
        hits = _glob.glob(os.path.join(folder, pattern), recursive=True)
        return hits[0] if hits else None

    # ── Long-MR-MS ────────────────────────────────────────────────────────
    long_path = os.path.join(raw_dir, 'Long-MR-MS')
    if os.path.exists(long_path):
        for p_folder in _glob.glob(os.path.join(long_path, 'patient*')):
            gt     = _glob.glob(os.path.join(p_folder, '*_gt.nii.gz'))
            flairs = _glob.glob(os.path.join(p_folder, '*_FLAIRreg.nii.gz'))
            if not (gt and flairs):
                continue
            for flair in flairs:
                if ablation == "A":
                    data_list.append({'image': flair, 'label': gt[0]})
                else:
                    t1 = _find_modality(p_folder, '*_T1reg.nii.gz') or \
                         _find_modality(p_folder, '*_T1*.nii.gz')
                    t2 = _find_modality(p_folder, '*_T2reg.nii.gz') or \
                         _find_modality(p_folder, '*_T2*.nii.gz')
                    if ablation == "B" and t1:
                        data_list.append({'image': [flair, t1], 'label': gt[0]})
                    elif ablation == "C" and t1 and t2:
                        data_list.append({'image': [flair, t1, t2], 'label': gt[0]})

    # ── Mendeley ──────────────────────────────────────────────────────────
    mendeley_path = os.path.join(raw_dir, 'Mendeley')
    if os.path.exists(mendeley_path):
        flair_imgs = [f for f in _glob.glob(os.path.join(mendeley_path, '**/*-Flair.nii'), recursive=True)
                      if "LESIONSEG" not in f.upper()]
        for f in flair_imgs:
            p_id = os.path.basename(f).split('-')[0]
            d    = os.path.dirname(f)
            mask = os.path.join(d, f"{p_id}-LesionSeg-Flair.nii")
            if not os.path.exists(mask):
                continue
            if ablation == "A":
                data_list.append({'image': f, 'label': mask})
            else:
                t1 = os.path.join(d, f"{p_id}-T1.nii")
                t2 = os.path.join(d, f"{p_id}-T2.nii")
                if ablation == "B" and os.path.exists(t1):
                    data_list.append({'image': [f, t1], 'label': mask})
                elif ablation == "C" and os.path.exists(t1) and os.path.exists(t2):
                    data_list.append({'image': [f, t1, t2], 'label': mask})

    # ── MSLesSeg ──────────────────────────────────────────────────────────
    msles_path = os.path.join(raw_dir, 'MSLesSeg')
    if os.path.exists(msles_path):
        all_nii  = _glob.glob(os.path.join(msles_path, '**/*.nii*'), recursive=True)
        mask_map = {}
        for m in all_nii:
            if "_MASK" in m.upper():
                parts = os.path.basename(m).split('_')
                if len(parts) >= 2:
                    mask_map[f"{parts[0]}_{parts[1]}"] = m
        flairs = [f for f in all_nii if "FLAIR" in f.upper() and "_MASK" not in f.upper()]
        for f in flairs:
            parts = os.path.basename(f).split('_')
            if len(parts) < 2:
                continue
            key = f"{parts[0]}_{parts[1]}"
            if key not in mask_map:
                continue
            m = mask_map[key]
            if ablation == "A":
                data_list.append({'image': f, 'label': m})
            else:
                base = os.path.dirname(f)
                prefix = '_'.join(parts[:2])
                t1 = next((_f for _f in all_nii if "_T1" in _f.upper() and prefix in _f and "_MASK" not in _f.upper()), None)
                t2 = next((_f for _f in all_nii if "_T2" in _f.upper() and prefix in _f and "_MASK" not in _f.upper()), None)
                if ablation == "B" and t1:
                    data_list.append({'image': [f, t1], 'label': m})
                elif ablation == "C" and t1 and t2:
                    data_list.append({'image': [f, t1, t2], 'label': m})

    n_ch = {"A": 1, "B": 2, "C": 3}[ablation]
    print(f"✅ Ablation {ablation} | {n_ch} channel(s) | {len(data_list)} pairs found")
    return data_list, n_ch

# Replace train_files with multi-modal version
train_files, N_CHANNELS = create_multimodal_data_list(DATA_DIR)

## 🚀 1.6 The Hardware-Agnostic Training Engine

In [ ]:
from monai.data import Dataset, DataLoader, pad_list_data_collate

def pad_list_collate_clone(batch):
    """Wrapper that clones all tensors before collating."""
    # RandCropByPosNegLabeld with num_samples>1 returns list-of-dicts per item;
    # flatten so batch is always a flat list of dicts.
    flat = []
    for b in batch:
        if isinstance(b, list):
            flat.extend(b)
        else:
            flat.append(b)
    for b in flat:
        for k, v in b.items():
            if hasattr(v, "as_tensor"):
                b[k] = v.as_tensor().clone()
            elif isinstance(v, torch.Tensor):
                b[k] = v.clone()
    return pad_list_data_collate(flat)

from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, SpatialPadd,
                               RandCropByPosNegLabeld, RandFlipd, RandRotate90d,
                               NormalizeIntensityd, ToTensord, ConcatItemsd,
                               ResizeWithPadOrCropd)
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy.ndimage import label as cc_label
import numpy as np, time

# ─── Hyperparameters ────────────────────────────────────────────────────────────────────────
EPOCHS        = 100
BATCH_SIZE    = 2    # RTX 4500 Ada 25.76 GB; with num_samples=4 gives 8 patches/step
PATCH_SIZE    = (128, 128, 128)
LR            = 1e-4
GRAD_ACCUM    = 4    # effective batch = 2*4 = 8, matching prior A100 training
SAVE_EVERY    = 5
# ──────────────────────────────────────────────────────────────────────────────

def _lesion_tpr(pred_bin, gt_bin):
    """
    Lesion-wise True Positive Rate: fraction of GT lesions that have at least
    one predicted voxel overlapping them. More sensitive to small lesions than Dice.
    Also returns counts split by lesion size (small <10, medium 10-100, large >100 voxels).
    """
    gt_cc, n_gt = cc_label(gt_bin)
    if n_gt == 0:
        return None, {}   # no lesions in this volume

    size_tp  = {"small": [0, 0], "medium": [0, 0], "large": [0, 0]}  # [tp, total]
    for comp_id in range(1, n_gt + 1):
        comp_mask = (gt_cc == comp_id)
        size      = int(comp_mask.sum())
        bucket    = "small" if size < 10 else ("medium" if size <= 100 else "large")
        size_tp[bucket][1] += 1
        if (pred_bin[comp_mask] > 0).any():
            size_tp[bucket][0] += 1

    total_tp    = sum(v[0] for v in size_tp.values())
    lesion_tpr  = total_tp / n_gt
    return lesion_tpr, size_tp


def _build_transforms(patch_size, n_channels, train=True):
    """Build MONAI transform pipeline.
    ToTensord is placed AFTER all spatial transforms to preserve MetaTensor metadata
    through SpatialPadd and RandCropByPosNegLabeld — moving it earlier breaks crop
    geometry on certain volume shapes, producing H=0 or D=75 crops.
    """
    img_keys = [f"image_{i}" for i in range(n_channels)] if n_channels > 1 else ["image"]
    load_keys = img_keys + ["label"]

    if n_channels > 1:
        load_tf = [
            LoadImaged(keys=load_keys),
            EnsureChannelFirstd(keys=load_keys),
            ConcatItemsd(keys=img_keys, name="image", dim=0),
            # NOTE: ToTensord intentionally omitted here — added at end of pipeline
        ]
    else:
        load_tf = [
            LoadImaged(keys=["image", "label"]),
            EnsureChannelFirstd(keys=["image", "label"]),
            # NOTE: ToTensord intentionally omitted here — added at end of pipeline
        ]

    base_tf = load_tf + [NormalizeIntensityd(keys=["image"], nonzero=True)]

    if train:
        return Compose(base_tf + [
            SpatialPadd(keys=["image", "label"], spatial_size=patch_size),
            RandCropByPosNegLabeld(keys=["image", "label"], label_key="label",
                                    spatial_size=patch_size, pos=1, neg=1, num_samples=4),
            ResizeWithPadOrCropd(keys=["image", "label"], spatial_size=patch_size),  # hard clamp to exact size
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
            RandRotate90d(keys=["image", "label"], prob=0.5),
            ToTensord(keys=["image", "label"]),  # AFTER all spatial transforms
        ])
    else:
        return Compose(base_tf + [
            ToTensord(keys=["image", "label"]),  # AFTER all spatial transforms
        ])


def _remap_keys(data_list, n_channels):
    """
    If multi-channel, split 'image' list into 'image_0', 'image_1', ... keys
    so each modality can be loaded independently by LoadImaged.
    """
    if n_channels == 1:
        return data_list
    out = []
    for d in data_list:
        entry = {"label": d["label"]}
        imgs  = d["image"] if isinstance(d["image"], list) else [d["image"]]
        for i, path in enumerate(imgs):
            entry[f"image_{i}"] = path
        out.append(entry)
    return out


def train_model(data_list, val_fraction=0.15):
    n_val        = max(1, int(len(data_list) * val_fraction))
    train_data   = _remap_keys(data_list[n_val:], N_CHANNELS)
    val_data     = _remap_keys(data_list[:n_val],  N_CHANNELS)

    train_tf = _build_transforms(PATCH_SIZE, N_CHANNELS, train=True)
    val_tf   = _build_transforms(PATCH_SIZE, N_CHANNELS, train=False)

    # Plain Dataset: preprocessed NIfTIs are already on disk, transforms run fast per-batch.
    # Eliminates PersistentDataset/MetaTensor cache interaction bugs entirely.
    train_ds = Dataset(data=train_data, transform=train_tf)
    val_ds   = Dataset(data=val_data,   transform=val_tf)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                              persistent_workers=False,   # True caused hangs on Windows spawn
                              prefetch_factor=2 if NUM_WORKERS > 0 else None,
                              collate_fn=pad_list_collate_clone)
    val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False,
                              num_workers=NUM_WORKERS,
                              pin_memory=(device.type == "cuda"),
                              collate_fn=pad_list_collate_clone)

    model       = get_model().to(device)
    if device.type == "cuda" and hasattr(torch, "compile") and sys.platform != "win32":
        model = torch.compile(model, mode="reduce-overhead")  # 20-30% speedup on PyTorch 2.0+ (Linux only)
    loss_fn     = MSLesionLoss().to(device)
    optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler   = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler      = torch.amp.GradScaler('cuda') if device.type == "cuda" else None
    dice_metric = DiceMetric(include_background=False, reduction="mean")

    best_dice = 0.0
    best_path = os.path.join(MODEL_SAVE_DIR, "best_model.pth")
    ckpt_path = os.path.join(MODEL_SAVE_DIR, "checkpoint_latest.pth")

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss, t0 = 0.0, time.time()
        optimizer.zero_grad()

        for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]")):
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            if scaler:
                with torch.amp.autocast('cuda'):
                    loss = loss_fn(model(images), labels) / GRAD_ACCUM
                scaler.scale(loss).backward()
                if (step + 1) % GRAD_ACCUM == 0:
                    scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            else:
                loss = loss_fn(model(images), labels) / GRAD_ACCUM
                loss.backward()
                if (step + 1) % GRAD_ACCUM == 0:
                    optimizer.step(); optimizer.zero_grad()
            epoch_loss += loss.item() * GRAD_ACCUM

        scheduler.step()

        # ── Validation ──────────────────────────────────────────────────────────────────
        model.eval()
        tpr_list   = []
        size_accum = {"small": [0, 0], "medium": [0, 0], "large": [0, 0]}

        from concurrent.futures import ThreadPoolExecutor as _TPE
        tpr_futures = []
        with _TPE(max_workers=1) as tpe:
            with torch.no_grad():
                for val_batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]", leave=False):
                    val_img = val_batch["image"].to(device)
                    val_lbl = val_batch["label"].to(device)
                    pred    = sliding_window_inference(val_img, PATCH_SIZE, 4, model)

                    pred_bin = (pred.sigmoid() > 0.5).float()
                    dice_metric(y_pred=pred_bin, y=val_lbl)

                    # Submit CPU connected-component work to thread so GPU starts next volume immediately
                    pred_np = pred_bin.squeeze().cpu().numpy().astype(bool)
                    gt_np   = val_lbl.squeeze().cpu().numpy().astype(bool)
                    tpr_futures.append(tpe.submit(_lesion_tpr, pred_np, gt_np))

            # Collect TPR results after GPU loop finishes
            for fut in tpr_futures:
                tpr, sz = fut.result()
                if tpr is not None:
                    tpr_list.append(tpr)
                    for k in size_accum:
                        size_accum[k][0] += sz[k][0]
                        size_accum[k][1] += sz[k][1]

        val_dice    = dice_metric.aggregate().item()
        dice_metric.reset()
        mean_tpr    = np.mean(tpr_list) if tpr_list else float('nan')

        def _tpr_str(k):
            tp, tot = size_accum[k]
            return f"{tp}/{tot}" if tot > 0 else "n/a"

        elapsed = (time.time() - t0) / 60
        print(f"Epoch {epoch+1:3d} | Loss: {epoch_loss/len(train_loader):.4f} | "
              f"Dice: {val_dice:.4f} | TPR: {mean_tpr:.3f} | "
              f"S:{_tpr_str('small')} M:{_tpr_str('medium')} L:{_tpr_str('large')} | "
              f"{elapsed:.1f}m")

        # ── Checkpoints ────────────────────────────────────────────────────────────────
        import threading
        def _save_async(state, path):
            torch.save(state, path)
        threading.Thread(
            target=_save_async,
            args=({"epoch": epoch, "model": model.state_dict(),
                   "optimizer": optimizer.state_dict(), "best_dice": best_dice}, ckpt_path),
            daemon=True
        ).start()  # non-blocking: next epoch starts without waiting for disk I/O
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), best_path)
            print(f"  New best Dice: {best_dice:.4f} -> saved to {best_path}")
        elif (epoch + 1) % SAVE_EVERY == 0:
            ep_path = os.path.join(MODEL_SAVE_DIR, f"model_epoch{epoch+1}.pth")
            torch.save(model.state_dict(), ep_path)

    print(f"\nTraining complete. Best Val Dice: {best_dice:.4f}")
    return model

print("Training engine ready.")
# train_model(processed_train_files)

In [ ]:
import subprocess

RCLONE_REMOTE     = "gdrive"              # rclone remote name (configured once)
DRIVE_MODELS_PATH = "MS_Project/models"   # path inside your Drive

def sync_models_to_drive():
    src = os.path.abspath(MODEL_SAVE_DIR)
    dst = f"{RCLONE_REMOTE}:{DRIVE_MODELS_PATH}"
    print(f"📤 Syncing {src} → {dst} ...")
    result = subprocess.run(["rclone", "copy", src, dst, "--progress"])
    if result.returncode == 0:
        print("✅ Sync complete — model is now in your Drive.")
    else:
        print("❌ rclone sync failed. Check rclone is installed and configured.")

if ENVIRONMENT == "local":
    sync_models_to_drive()

## ☁️ 1.7 rclone Sync to Google Drive

Syncs `./models/` to Drive after training. Requires a one-time setup:

```bash
# 1. Download rclone (no install needed — just a binary)
#    Windows: https://rclone.org/downloads/
#    Linux:   curl https://rclone.org/install.sh | sudo bash

# 2. Configure once on home machine (opens browser for Google sign-in)
rclone config
# → New remote → name: "gdrive" → type: Google Drive → follow prompts

# 3. Copy config to the university machine (no browser needed there)
#    Windows: C:\Users\<you>\AppData\Roaming\rclone\rclone.conf
#    Linux:   ~/.config/rclone/rclone.conf
```